# Fake News Detector 
Using LIAR dataset and a Random Forest Classifier

# Imports

In [1]:
import pandas as pd
import numpy as np
import re
import string
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, train_test_split, StratifiedKFold, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from scipy.sparse import csr_matrix, hstack
import warnings
import joblib
warnings.filterwarnings("ignore")

# Loading Dataset and Combining Train, Test and Validation tsv's

In [2]:
train_df = pd.read_csv('dataset/train.tsv', sep='\t', header=None)
val_df = pd.read_csv('dataset/valid.tsv', sep='\t', header=None)
test_df = pd.read_csv('dataset/test.tsv', sep='\t', header=None)

column_names = ['id', 'label', 'statement', 'subject', 'speaker',
                'job_title', 'state_info', 'party_affiliation',
                'barely_true_counts', 'false_counts', 'half_true_counts',
                'mostly_true_counts', 'pants_fire_counts', 'context']

train_df.columns = column_names
val_df.columns = column_names
test_df.columns = column_names

combined_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
combined_df.drop(columns=["id"], inplace=True)
print(combined_df.head())


         label                                          statement  \
0        false  Says the Annies List political group supports ...   
1    half-true  When did the decline of coal start? It started...   
2  mostly-true  Hillary Clinton agrees with John McCain "by vo...   
3        false  Health care reform legislation is likely to ma...   
4    half-true  The economic turnaround started at the end of ...   

                              subject         speaker             job_title  \
0                            abortion    dwayne-bohac  State representative   
1  energy,history,job-accomplishments  scott-surovell        State delegate   
2                      foreign-policy    barack-obama             President   
3                         health-care    blog-posting                   NaN   
4                        economy,jobs   charlie-crist                   NaN   

  state_info party_affiliation  barely_true_counts  false_counts  \
0      Texas        republican            

# Clean Text Function

In [3]:
def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r'https?://\S+|www\.\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'<.*?>+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    text = re.sub(r'[^\w\s\']', ' ', text)
    text = re.sub(r'\b\d+\b', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# Preprocessing

In [4]:
combined_df['statement'] = combined_df['statement'].apply(clean_text) # Clean text for statements

Subject column split by commas to indicate different subjects, splitting into separate columns

In [5]:
combined_df["subject"] = combined_df["subject"].str.split(",")
combined_df = combined_df.explode("subject")

Label encoding all the categorical columns

In [10]:
categorical_features = ['speaker', 'job_title', 'state_info', 'party_affiliation', 'subject', 'context']
categorical_encoders = {}
for col in categorical_features:
    categorical_encoders[col] = LabelEncoder()
    combined_df[col] = categorical_encoders[col].fit_transform(combined_df[col])
    joblib.dump(categorical_encoders[col], f'{col}_encoder.pkl')

Use min max scaling on numerical features

In [7]:
num_cols = ['barely_true_counts', 'false_counts', 'half_true_counts',
                     'mostly_true_counts', 'pants_fire_counts']

X_numerical = combined_df[num_cols].fillna(0)

# Apply Min-Max scaling
scaler = MinMaxScaler()
for col in num_cols:
    combined_df[col] = scaler.fit_transform(combined_df[[col]])   


In [8]:
vectorizer = TfidfVectorizer(
    max_features=15000
)
X = vectorizer.fit_transform(combined_df["statement"])
joblib.dump(vectorizer, 'tfidf_vectorizer.joblib')  

y = combined_df['label'].values

# Split data using train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.1, random_state=42
)

Setup base random forest classifier, stratified k fold, param grid for hyper param tuning and Randomized CV Search

In [ ]:
model = RandomForestClassifier()

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

param_grid = {
        'n_estimators': [100],
        'max_depth': [None],
        'min_samples_split': [10],
        'min_samples_leaf': [1],
        'max_features': ['log2']
    }

grid_search = RandomizedSearchCV(
        estimator=model,
        param_distributions=param_grid,
        n_iter=100,
        cv=skf,
        scoring='accuracy',
        n_jobs=-1,
        verbose=1
)

Fit and find the best params / cv scores 

In [22]:
from joblib import parallel_backend
with parallel_backend('loky', n_jobs=-1):
    grid_search.fit(X, y)

print("Best Parameters:", grid_search.best_params_)

print("Best cross-validation score:", grid_search.best_score_)

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best Parameters: {'n_estimators': 100, 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': None}
Best cross-validation score: 0.8188781643405738


Fit the best model using training data

In [23]:
import joblib


best_model = grid_search.best_estimator_
best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_test)

print("=== TEST SET EVALUATION ===")
print("Accuracy:", accuracy_score(y_test, y_pred))      
print("F1 Score:", f1_score(y_test, y_pred, average='weighted'))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

joblib.dump(best_model, 'best_rfc.pkl')

=== TEST SET EVALUATION ===
Accuracy: 0.8504875406283857
F1 Score: 0.8510621689815793

Classification Report:
               precision    recall  f1-score   support

 barely-true       0.92      0.87      0.90       490
       false       0.79      0.89      0.84       528
   half-true       0.79      0.88      0.83       566
 mostly-true       0.84      0.86      0.85       523
  pants-fire       0.99      0.76      0.86       259
        true       0.90      0.79      0.84       403

    accuracy                           0.85      2769
   macro avg       0.87      0.84      0.85      2769
weighted avg       0.86      0.85      0.85      2769



['best_rfc.pkl']